El trabajo se realizo en colaboracion con Claude

In [ ]:
import random

nombres = ["Ana", "Luis", "Carlos", "María", "Jorge", "Laura", "Pedro", "Diana",
           "Andrés", "Paola", "Felipe", "Natalia", "Julián", "Sara", "Tomás"]

def generar_estudiante(id):
    nombre   = random.choice(nombres)
    edad     = random.randint(16, 30)
    promedio = round(random.uniform(0.0, 5.0), 1)
    return {"id": id, "nombre": nombre, "edad": edad, "promedio": promedio}

ids = list(range(10000))
random.shuffle(ids)
estudiantes = [generar_estudiante(i) for i in ids]

In [ ]:
for e in estudiantes[:5]:
    print(e)

{'id': 6480, 'nombre': 'Jorge', 'edad': 24, 'promedio': 0.7}
{'id': 6338, 'nombre': 'Andrés', 'edad': 29, 'promedio': 4.3}
{'id': 4806, 'nombre': 'Pedro', 'edad': 30, 'promedio': 1.9}
{'id': 4464, 'nombre': 'Paola', 'edad': 30, 'promedio': 1.3}
{'id': 7662, 'nombre': 'Tomás', 'edad': 27, 'promedio': 0.7}


In [ ]:
def buscar_por_id(estudiantes, id):
    for estudiante in estudiantes:
        if estudiante["id"] == id:
            return estudiante
    return None

In [ ]:
print(buscar_por_id(estudiantes, 50))

{'id': 50, 'nombre': 'Andrés', 'edad': 18, 'promedio': 2.9}


In [ ]:
class NodoABB:
    def __init__(self, estudiante):
        self.estudiante = estudiante
        self.izq = None
        self.der = None

class ABB:
    def __init__(self):
        self.raiz = None

    def insertar(self, estudiante):
        self.raiz = self._insertar(self.raiz, estudiante)

    def _insertar(self, nodo, estudiante):
        if nodo is None:
            return NodoABB(estudiante)
        if estudiante["id"] < nodo.estudiante["id"]:
            nodo.izq = self._insertar(nodo.izq, estudiante)
        elif estudiante["id"] > nodo.estudiante["id"]:
            nodo.der = self._insertar(nodo.der, estudiante)
        return nodo

    def buscarPorID(self, id):
        return self._buscar(self.raiz, id)

    def _buscar(self, nodo, id):
        if nodo is None:
            return None
        if id == nodo.estudiante["id"]:
            return nodo.estudiante
        elif id < nodo.estudiante["id"]:
            return self._buscar(nodo.izq, id)
        else:
            return self._buscar(nodo.der, id)

In [ ]:
arbol = ABB()
for e in estudiantes:
    arbol.insertar(e)

In [ ]:
print(arbol.buscarPorID(50))

{'id': 50, 'nombre': 'Andrés', 'edad': 18, 'promedio': 2.9}


In [ ]:
import bisect

class ArbolBPlus:
    """
    Árbol B+ simplificado.

    Idea principal:
      - Todos los datos viven en 'páginas' (bloques de tamaño fijo).
      - Las páginas están ordenadas y enlazadas entre sí.
      - Un índice guarda el primer id de cada página para navegar rápido.

    Así la búsqueda tiene dos pasos:
      1. Búsqueda binaria en el índice → encontrar la página correcta.
      2. Búsqueda binaria dentro de la página → encontrar el estudiante.

    Complejidad: O(log n) gracias a la doble búsqueda binaria.
    """

    def __init__(self, datos, tam_pagina=50):
        """
        Construye el árbol al instanciar la clase.
        tam_pagina define cuántos estudiantes caben en cada página (hoja).
        """
        self.tam_pagina = tam_pagina
        self.paginas = []  # Lista de páginas (hojas)
        self.indice  = []  # Lista de los ids mínimos de cada página
        self._construir(datos)

    def _construir(self, datos):
        """
        Ordena los datos y los divide en páginas de tamaño fijo.
        Luego construye el índice con el primer id de cada página.
        """
        ordenados = sorted(datos, key=lambda x: x['id'])

        # Dividir en páginas del tamaño definido
        for i in range(0, len(ordenados), self.tam_pagina):
            pagina = ordenados[i : i + self.tam_pagina]
            self.paginas.append(pagina)

        # El índice guarda el id mínimo de cada página
        # Ejemplo: [0, 50, 100, 150, ...] → permite saber en qué página buscar
        self.indice = [pagina[0]['id'] for pagina in self.paginas]

    def buscar_por_id(self, target_id):
        """
        Busca un estudiante en dos pasos:
          1. bisect_right sobre el índice → número de página candidata.
          2. bisect_left dentro de esa página → posición exacta del registro.
        """
        if not self.indice:
            return None

        # Paso 1: encontrar en qué página debería estar el id
        # bisect_right devuelve la posición de inserción, restamos 1 para la página anterior
        num_pagina = bisect.bisect_right(self.indice, target_id) - 1
        num_pagina = max(0, num_pagina)  # Nunca menor que 0
        pagina = self.paginas[num_pagina]

        # Paso 2: búsqueda binaria dentro de la página
        ids_en_pagina = [e['id'] for e in pagina]
        pos = bisect.bisect_left(ids_en_pagina, target_id)

        # Verificar que la posición encontrada corresponde exactamente al id buscado
        if pos < len(pagina) and pagina[pos]['id'] == target_id:
            return pagina[pos]

        return None  # No encontrado

# Construimos el árbol B+ con páginas de 50 estudiantes cada una
arbol_bp = ArbolBPlus(estudiantes, tam_pagina=50)

print(arbol_bp.buscar_por_id(8555))

{'id': 9805, 'nombre': 'Natalia', 'edad': 30, 'promedio': 0.2}


In [ ]:
import time
import random

# Número de búsquedas a realizar por cada estructura.
# 100.000 es seguro en Colab para ABB y B+ (milisegundos).
# Para la lista puede tomar ~30-60 segundos — si quieres
# reducirlo baja este valor a 10000.
ITERACIONES = 100_000

# Generamos los ids a buscar una sola vez para que
# las tres estructuras busquen exactamente los mismos valores
ids_a_buscar = [random.randint(0, 9999) for _ in range(ITERACIONES)]

In [ ]:
t_inicio = time.process_time()

for target in ids_a_buscar:
    buscar_por_id(estudiantes, target)

t_fin = time.process_time()

tiempo_lista       = t_fin - t_inicio
promedio_lista     = tiempo_lista / ITERACIONES

print(f"📋 BENCHMARK LISTA")
print(f"-----------------------------------")
print(f"Iteraciones:                {ITERACIONES}")
print(f"Tiempo total CPU:           {tiempo_lista:.6f} s")
print(f"Tiempo promedio búsqueda:   {promedio_lista:.10f} s")

📋 BENCHMARK LISTA
-----------------------------------
Iteraciones:                100000
Tiempo total CPU:           25.604630 s
Tiempo promedio búsqueda:   0.0002560463 s


In [ ]:
t_inicio = time.process_time()

for target in ids_a_buscar:
    arbol.buscarPorID(target)

t_fin = time.process_time()

tiempo_abb         = t_fin - t_inicio
promedio_abb       = tiempo_abb / ITERACIONES

print(f"\n🌳 BENCHMARK ABB")
print(f"-----------------------------------")
print(f"Iteraciones:                {ITERACIONES}")
print(f"Tiempo total CPU:           {tiempo_abb:.6f} s")
print(f"Tiempo promedio búsqueda:   {promedio_abb:.10f} s")


🌳 BENCHMARK ABB
-----------------------------------
Iteraciones:                100000
Tiempo total CPU:           0.321423 s
Tiempo promedio búsqueda:   0.0000032142 s


In [ ]:
t_inicio = time.process_time()

for target in ids_a_buscar:
    arbol_bp.buscar_por_id(target)

t_fin = time.process_time()

tiempo_bplus       = t_fin - t_inicio
promedio_bplus     = tiempo_bplus / ITERACIONES

print(f"\n🌲 BENCHMARK B+")
print(f"-----------------------------------")
print(f"Iteraciones:                {ITERACIONES}")
print(f"Tiempo total CPU:           {tiempo_bplus:.6f} s")
print(f"Tiempo promedio búsqueda:   {promedio_bplus:.10f} s")


🌲 BENCHMARK B+
-----------------------------------
Iteraciones:                100000
Tiempo total CPU:           0.537655 s
Tiempo promedio búsqueda:   0.0000053766 s


In [ ]:
print(f"\n{'='*40}")
print(f"{'COMPARATIVA FINAL':^40}")
print(f"{'='*40}")
print(f"ABB  es {tiempo_lista/tiempo_abb:.1f}x más rápido que Lista")
print(f"B+   es {tiempo_lista/tiempo_bplus:.1f}x más rápido que Lista")
print(f"B+   es {tiempo_abb/tiempo_bplus:.1f}x más rápido que ABB")


           COMPARATIVA FINAL            
ABB  es 79.7x más rápido que Lista
B+   es 47.6x más rápido que Lista
B+   es 0.6x más rápido que ABB
